# whisper-small LoRA 파인튜닝 — 608 스모크 검증

**목적**: 파이프라인이 Colab T4에서 에러 없이 도는지 확인(소량 2화자 데이터).
학습 품질은 무의미 — "노트북이 끝까지 돈다"만 본다.

**전제**: 런타임 → 런타임 유형 변경 → **T4 GPU**. 그리고 `colab_trainset.zip`을 드라이브에 업로드.


## 셀 1 — GPU 확인 (Tesla T4가 떠야 함)

In [ ]:
!nvidia-smi

## 셀 2 — 설치 (torchao 제거 + numpy 고정) → 실행 후 [런타임 → 세션 다시 시작] 필수
numpy 버전을 **손으로 적지 않는다**. Colab 이미지의 torch/librosa는 그 이미지의 numpy에 맞춰 빌드돼 있으므로,
지금 런타임의 numpy 버전을 읽어서 그대로 못박는다. 재시작 후에는 셀 2를 건너뛰고 **셀 2.5**부터.

> ⚠️ 셀 2는 **초기화된 런타임**에서 실행해야 한다. 이미 `pip -U`를 한 번 돌린 세션이면 numpy가 이미 올라가 있어서
> 잘못된 버전을 고정하게 된다 → 먼저 [런타임 → 세션 삭제 후 다시 시작].


In [ ]:
# Colab 이미지가 들고 있는 numpy 버전을 읽어서 그대로 고정한다.
# 숫자를 하드코딩하면(예전의 numpy==2.0.2) Colab 이미지가 바뀔 때마다 다시 깨진다:
#   numpy가 너무 높음 → AttributeError: module 'numpy' has no attribute '_no_nep50_warning'
#   numpy가 너무 낮음 → ImportError: cannot import name '_center' from numpy._core.umath
import numpy
NPV = numpy.__version__
print("Colab 기본 numpy:", NPV, "→ 이 버전으로 고정")

# 핵심: numpy 핀을 같은 pip 호출 안에 넣어야 resolver가 도로 올리지 못한다.
!pip install -q -U transformers datasets peft accelerate evaluate jiwer librosa "numpy=={NPV}"
!pip uninstall -y -q torchao   # peft 버전검사와 충돌, LoRA엔 불필요

!pip show numpy | grep -i ^Version    # 디스크에 실제로 깔린 버전 = NPV여야 정상
print("✅ 설치 완료 → [런타임 → 세션 다시 시작] 반드시! → 셀 2.5부터")

## 셀 2.5 — 재시작 후 검증 (여기서 통과해야 아래가 의미 있음)
버전 3종을 찍고 실제 임포트까지 해본다. 실패하면 아래 셀들은 전부 같은 이유로 터지니 여기서 잡는다.
- `numpy has no attribute '_no_nep50_warning'` → 런타임 torch가 지금 numpy(≥2.3)를 못 따라감
- `cannot import name '_center' from numpy._core.umath` → 반대로 numpy가 너무 낮음

**처방은 둘 다 동일**: 런타임 → **세션 삭제 후 다시 시작** → 셀 2부터 다시(셀 2가 이미지 기준 numpy를 유지한다).
버전 숫자를 손으로 고쳐 맞추려 하지 말 것.

In [ ]:
import numpy, torch, transformers
print("numpy       :", numpy.__version__)
print("torch       :", torch.__version__)
print("transformers:", transformers.__version__)

# torch↔numpy 불일치면 바로 아래 줄에서 터진다(셀 4의 그 에러와 같은 원인)
from transformers import WhisperForConditionalGeneration
import librosa; librosa.stft(numpy.zeros(2048, dtype=numpy.float32))   # _center 계열 조기 탐지
print("✅ 임포트 정상 — 셀 3으로")

## 셀 3 — 드라이브 마운트 + 업로드한 zip 풀기 + 경로 확인
`ZIP`을 본인이 올린 위치로 수정.

In [ ]:
import os, torch, json
from google.colab import drive
drive.mount('/content/drive')

ZIP  = "/content/drive/MyDrive/colab_trainset.zip"   # ← 업로드한 zip 경로
BASE = "/content/608-trainset"                       # 풀 위치(세션 로컬, 빠름)
if os.path.exists(ZIP):
    !unzip -oq "$ZIP" -d "$BASE"
else:
    # zip 대신 폴더째 올렸으면 그 경로를 BASE로 지정하고 위 unzip은 건너뛴다
    BASE = "/content/drive/MyDrive/608-trainset"

print("CUDA:", torch.cuda.is_available())
for f in ["train.jsonl","test.jsonl"]:
    p=os.path.join(BASE,f); n=sum(1 for _ in open(p,encoding="utf-8"))
    print(f"  {f}: {n}개")
print("wav:", sum(len(fs) for _,_,fs in os.walk(os.path.join(BASE,'wav'))), "개")

## 셀 4 — 모델 + LoRA

In [ ]:
from transformers import WhisperForConditionalGeneration, WhisperProcessor
from peft import LoraConfig, get_peft_model

processor = WhisperProcessor.from_pretrained("openai/whisper-small", language="ko", task="transcribe")
model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small")
model.config.forced_decoder_ids = None
model.config.suppress_tokens = []
model.config.use_cache = False

model = get_peft_model(model, LoraConfig(
    r=16, lora_alpha=32, target_modules=["q_proj","v_proj"], lora_dropout=0.05))
model.print_trainable_parameters()

# 생성(평가) 설정 — 이거 없으면 whisper가 반복 환각으로 CER이 2.0+로 부풀려진다.
model.generation_config.language = "ko"
model.generation_config.task = "transcribe"
model.generation_config.no_repeat_ngram_size = 3   # 같은 구절 반복 억제
model.generation_config.max_new_tokens = 100

## 셀 5 — 데이터셋 로드 + 전처리

In [ ]:
from datasets import load_dataset, Audio
ds = load_dataset("json", data_files={
    "train": f"{BASE}/train.jsonl", "test": f"{BASE}/test.jsonl"})
ds = ds.map(lambda b: {"audio": os.path.join(BASE, b["audio"])})
ds = ds.cast_column("audio", Audio(sampling_rate=16000))

def prep(b):
    a = b["audio"]
    b["input_features"] = processor.feature_extractor(
        a["array"], sampling_rate=16000).input_features[0]
    b["labels"] = processor.tokenizer(b["text"]).input_ids
    return b
ds = ds.map(prep, remove_columns=ds["train"].column_names)
print(ds)

## 셀 6 — 데이터 콜레이터 (whisper 필수)

In [ ]:
from dataclasses import dataclass
from typing import Any
@dataclass
class Collator:
    processor: Any
    def __call__(self, feats):
        batch = self.processor.feature_extractor.pad(
            [{"input_features": f["input_features"]} for f in feats], return_tensors="pt")
        labels = self.processor.tokenizer.pad(
            [{"input_ids": f["labels"]} for f in feats], return_tensors="pt")
        lab = labels["input_ids"].masked_fill(labels.attention_mask.ne(1), -100)
        if (lab[:,0] == self.processor.tokenizer.bos_token_id).all():
            lab = lab[:,1:]
        batch["labels"] = lab
        return batch
collator = Collator(processor)

## 셀 7 — CER 지표 (baseline 0.70 비교)

In [ ]:
import evaluate
cer_metric = evaluate.load("cer")
def compute_metrics(p):
    lab = p.label_ids
    lab[lab == -100] = processor.tokenizer.pad_token_id
    pred = processor.tokenizer.batch_decode(p.predictions, skip_special_tokens=True)
    ref  = processor.tokenizer.batch_decode(lab, skip_special_tokens=True)
    return {"cer": cer_metric.compute(predictions=pred, references=ref)}

## 셀 8 — 학습
- `learning_rate=1e-4` (1e-3은 과학습/발산). 데이터 많아지면 `num_train_epochs`는 3~5로.
- 실제(대규모) 학습은 `output_dir`을 드라이브로(체크포인트 재개).

In [ ]:
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments
args = Seq2SeqTrainingArguments(
    output_dir="/content/whisper-608-lora",    # 대규모 학습은 /content/drive/... 로
    per_device_train_batch_size=8,
    learning_rate=1e-4, warmup_steps=20,       # LoRA whisper 표준 LR
    num_train_epochs=4, fp16=True,
    eval_strategy="epoch", save_strategy="no", logging_steps=10,
    predict_with_generate=True, generation_max_length=225,
    remove_unused_columns=False, label_names=["labels"], report_to="none")
trainer = Seq2SeqTrainer(
    model=model, args=args,
    train_dataset=ds["train"], eval_dataset=ds["test"],
    data_collator=collator, compute_metrics=compute_metrics,
    processing_class=processor)

# 공정 비교: 파인튜닝 전 baseline(=원본 whisper) CER 먼저 찍고 학습.
print("파인튜닝 전 baseline:", trainer.evaluate())
trainer.train()

## 셀 9 — 결과 + 파이프라인 OK 확인

In [ ]:
m = trainer.evaluate()
print(f"파인튜닝 후 test CER {m['eval_cer']:.3f}")
print("→ 셀 8의 '전 baseline'과 비교: 후 < 전 이면 파인튜닝이 먹힌 것.")
print("→ 데이터가 적으면 과적합(암기)으로 개선 없음 — 화자·시간 규모를 키워라.")
# 저장(LoRA 어댑터만 수십MB)
model.save_pretrained(f"{BASE}/whisper-608-lora-final")